# 15 — Explainability with SHAP (Section 5.6)

Once the volatility model exists, SHAP values are nearly free and answer 'why did the model predict *this*?' for any single forecast. Two uses:

- **Dashboard** — turns a bare number into something a user can evaluate ('volatility is forecast high mainly because recent realised vol and ATR are elevated').
- **Debugging** — when a prediction looks wrong, its top SHAP features tell you where to look first.

SHAP is a validation/interpretation tool, not part of the serving hot path — TreeExplainer is fast but per-prediction attribution is an on-demand or dashboard-side computation.

In [9]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

warnings.filterwarnings("ignore")

ImportError: Numba needs NumPy 2.4 or less. Got NumPy 2.5.

In [ ]:


project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate)); break

from src.forecast_engine.forecasting.gbm_boost import VolatilityModel, TARGET_COLUMN
from src.forecast_engine.features.schema import FEATURE_NAMES

data = pd.read_parquet(project_root / "data" / "processed" / "sp500_features.parquet")
data["date"] = pd.to_datetime(data["date"])
feature_cols = [c for c in FEATURE_NAMES if c in data.columns and not data[c].isna().all()]
model_data = data.dropna(subset=feature_cols + [TARGET_COLUMN]).reset_index(drop=True)
print(f"{len(model_data):,} rows, {len(feature_cols)} features")

## Fit and build the explainer

In [ ]:
model = VolatilityModel(feature_cols, schema_version="2.2").fit(model_data)
booster = model._model            # the underlying XGBRegressor
explainer = shap.TreeExplainer(booster)

sample = model_data[feature_cols].sample(min(2000, len(model_data)), random_state=0)
shap_values = explainer.shap_values(sample)
print(f"SHAP values computed for {sample.shape[0]} rows x {sample.shape[1]} features")

## Global importance: what drives volatility forecasts overall

Mean absolute SHAP value per feature — the model's global explanation. For a validated volatility model this should be dominated by the volatility family (realised vol, ATR, Bollinger width), which is the leakage-free story notebook 08b established.

In [ ]:
mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols).sort_values(ascending=False)
print("Top 12 features by mean |SHAP|:")
print(mean_abs.head(12).round(5).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
mean_abs.head(12).iloc[::-1].plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("mean |SHAP value|"); ax.set_title("Global feature importance (SHAP)")
plt.tight_layout(); plt.show()

## Local explanation: why one specific forecast

For a single prediction, the SHAP values decompose it into per-feature contributions from the base value. This is the per-forecast explanation a dashboard would surface next to the number.

In [ ]:
idx = 0
row_shap = pd.Series(shap_values[idx], index=feature_cols).sort_values(key=np.abs, ascending=False)
base = float(explainer.expected_value)
prediction = base + shap_values[idx].sum()
print(f"base value (avg prediction): {base:.4f}")
print(f"this prediction            : {prediction:.4f}\n")
print("Top contributions (feature: SHAP, + pushes vol up):")
for feat, val in row_shap.head(8).items():
    print(f"  {feat:<32} {val:+.5f}")

## Conclusions

**Global:** if the volatility family dominates mean |SHAP|, the model's explanation matches the validated story from 08b — it forecasts volatility from volatility-related features, no leakage, no spurious driver. If some unexpected feature dominates, that is a flag to investigate (possible leakage or artefact).

**Local:** per-prediction SHAP is the dashboard's 'why' — it converts a forecast into an auditable explanation, and it is the first place to look when a prediction seems wrong.

SHAP does not change any number the engine produces; it makes the numbers *inspectable*, which supports the honest-framing principle (5.7): a modelled estimate a user can interrogate is more trustworthy than an opaque one.